In [ ]:
import json
import os


cwd = os.getcwd()
if os.path.basename(cwd) == 'src':
    project_root = os.path.dirname(cwd)
else:
    project_root = cwd

config_path = os.path.join(project_root, 'config', 'config.json')

with open(config_path, 'r') as f:
    config = json.load(f)


config['dataset']['root'] = os.path.join(project_root, config['dataset']['root'])
results_dir = os.path.join(project_root, 'results')


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(config['dataset']['mean'], config['dataset']['std'])
])

cifar10_full = torchvision.datasets.CIFAR10(root=config['dataset']['root'], train=True, download=True, transform=transform)

pool_loader = torch.utils.data.DataLoader(cifar10_full, batch_size=config['feature_extractor']['batch_size'], shuffle=False)

In [ ]:
# pretrained feature extractor
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.DEFAULT)
print(f"loaded standard ImageNet ResNet18")

model.fc = torch.nn.Identity() 
model = model.to(device)
model.eval() 

embeddings = []
true_labels = [] 

with torch.no_grad():
    for images, labels in pool_loader:
        images = images.to(device)
        
        features = model(images)
        # l2 normalise
        features_normalized = F.normalize(features, p=2, dim=1)
        
        embeddings.append(features_normalized.cpu().numpy())
        true_labels.append(labels.numpy())

embeddings = np.concatenate(embeddings, axis=0)
true_labels = np.concatenate(true_labels, axis=0)

print(f"embeddings shape: {embeddings.shape}") 

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(embeddings)
distances, indices = nn.kneighbors(embeddings)

k_distances = np.sort(distances[:, k-1])

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.axhline(y=0.67, color='r', linestyle='--', label='Tuned Threshold (0.67)')
plt.title(f'K-Distance Graph (k={k})')
plt.xlabel('Points (sorted by distance)')
plt.ylabel(f'{k}-th Nearest Neighbor Distance')
plt.legend()
plt.grid(True)
plt.show()

print(f"Median k-distance: {np.median(k_distances):.4f}")

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(embeddings)
distances, indices = nn.kneighbors(embeddings)

k_distances = np.sort(distances[:, k-1])

plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.axhline(y=0.67, color='r', linestyle='--', label='Tuned Threshold (0.67)')
plt.title(f'K-Distance Graph (k={k})')
plt.xlabel('Points (sorted by distance)')
plt.ylabel(f'{k}-th Nearest Neighbor Distance')
plt.legend()
plt.grid(True)
plt.show()

print(f"Median k-distance: {np.median(k_distances):.4f}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

def get_typiclust_queries(embeddings, B=50, max_clusters=500, noise_threshold=None, min_samples=None, pca_dims=None):
    
    if noise_threshold is None: noise_threshold = config['typiclust']['noise_threshold']
    if pca_dims is None: pca_dims = config['typiclust']['pca_dims']
    
    N = embeddings.shape[0]
    random_state = config['typiclust']['random_state']
    
    nn_outlier = NearestNeighbors(n_neighbors=5)
    nn_outlier.fit(embeddings)
    distances_outlier, _ = nn_outlier.kneighbors(embeddings)
    noise_scores = distances_outlier[:, -1] 
    
    valid_mask = noise_scores <= noise_threshold
    clean_embeddings = embeddings[valid_mask]
    original_indices = np.where(valid_mask)[0] 
    
    print(f"KNN-Distance removed {N - len(clean_embeddings)} outliers (beyond {noise_threshold} threshold).")
    
    pca = PCA(n_components=min(pca_dims, clean_embeddings.shape[1]))
    reduced_embeddings = pca.fit_transform(clean_embeddings)
    print(f"Reduced dimensions to {reduced_embeddings.shape[1]} using PCA.")
    
    #fallback
    if len(reduced_embeddings) < B:
        reduced_embeddings = embeddings
        original_indices = np.arange(N)
    
    num_clusters = min(B, max_clusters)
    print(f"clustering {len(reduced_embeddings)} clean embeddings into {num_clusters} clusters")
    
    if num_clusters <= 50:
        kmeans = KMeans(n_clusters=num_clusters, random_state=random_state, n_init=config['typiclust']['kmeans_n_init'])
    else:
        kmeans = MiniBatchKMeans(n_clusters=num_clusters, random_state=random_state, n_init=config['typiclust']['minibatch_kmeans_n_init'])
        
    cluster_labels = kmeans.fit_predict(reduced_embeddings)
    
    clusters = {i: [] for i in range(num_clusters)}
    for idx, label in enumerate(cluster_labels):
        clusters[label].append(idx)
   
    valid_clusters = {k: v for k, v in clusters.items() if len(v) >= config['typiclust']['min_cluster_size']}
    sorted_clusters = sorted(valid_clusters.items(), key=lambda item: len(item[1]), reverse=True)
    top_b_clusters = sorted_clusters[:B]
    
    queries = []
    print(f"calculating typicality across the top {len(top_b_clusters)} clusters using KNN")
    
    for cluster_id, indices in top_b_clusters:
        cluster_embeddings = reduced_embeddings[indices]
        
        nn_typicality = NearestNeighbors(n_neighbors=min(20, len(cluster_embeddings)))
        nn_typicality.fit(cluster_embeddings)
        distances_typical, _ = nn_typicality.kneighbors(cluster_embeddings)
        typicality = -np.mean(distances_typical, axis=1)
        
        best_local_idx = np.argmax(typicality)
        
        best_clean_idx = indices[best_local_idx]
        best_global_idx = original_indices[best_clean_idx]
        
        queries.append(best_global_idx)
        
    return queries

BUDGET = config['typiclust']['budget']
query_indices = get_typiclust_queries(embeddings, B=BUDGET, max_clusters=config['typiclust']['max_clusters'], noise_threshold=0.67)

print(f"{len(query_indices)} images to be labeled")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

X_train_labeled = embeddings[query_indices]
y_train_labeled = true_labels[query_indices]

testset = torchvision.datasets.CIFAR10(root=config['dataset']['root'], train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=config['classifier']['test_batch_size'], shuffle=False)

test_embeddings = []
test_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        features = model(images)
        features_normalized = F.normalize(features, p=2, dim=1) 
        
        test_embeddings.append(features_normalized.cpu().numpy())
        test_labels.append(labels.numpy())

X_test = np.concatenate(test_embeddings, axis=0)
y_test = np.concatenate(test_labels, axis=0)

train_dataset = TensorDataset(torch.tensor(X_train_labeled, dtype=torch.float32), 
                              torch.tensor(y_train_labeled, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), 
                             torch.tensor(y_test, dtype=torch.long))

train_dl = DataLoader(train_dataset, batch_size=BUDGET, shuffle=True)
test_dl = DataLoader(test_dataset, batch_size=config['classifier']['test_batch_size'], shuffle=False)

num_features = embeddings.shape[1]
linear_classifier = nn.Linear(num_features, config['dataset']['num_classes']).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(linear_classifier.parameters(), lr=config['classifier']['lr'], momentum=config['classifier']['momentum'])

linear_classifier.train()
epochs = config['classifier']['epochs']

for epoch in range(epochs):
    for inputs, targets in train_dl:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = linear_classifier(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

linear_classifier.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, targets in test_dl:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = linear_classifier(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

accuracy = 100 * correct / total
print(f"final test acc using typiclust ({BUDGET} labels): {accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import random

budgets = config['evaluation']['budgets']
typiclust_accuracies = []
random_accuracies = []

for b in budgets:
    tc_queries = get_typiclust_queries(embeddings, B=b, max_clusters=config['typiclust']['max_clusters'])

    rand_queries = random.sample(range(len(embeddings)), b)
    
    def eval_queries(queries):
        X_tr = embeddings[queries]
        y_tr = true_labels[queries]
        
        train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
        train_dl = DataLoader(train_ds, batch_size=b, shuffle=True)
        
        num_features = embeddings.shape[1]
        clf = nn.Linear(num_features, config['dataset']['num_classes']).to(device)
        opt = optim.SGD(clf.parameters(), lr=config['classifier']['lr'], momentum=config['classifier']['momentum'])
        
        clf.train()
        for _ in range(config['classifier']['epochs']):
            for inputs, targets in train_dl:
                inputs, targets = inputs.to(device), targets.to(device)
                opt.zero_grad()
                loss = criterion(clf(inputs), targets)
                loss.backward()
                opt.step()
                
        clf.eval()
        correct = 0
        with torch.no_grad():
            for inputs, targets in test_dl:
                inputs, targets = inputs.to(device), targets.to(device)
                _, pred = torch.max(clf(inputs).data, 1)
                correct += (pred == targets).sum().item()
        return 100 * correct / len(test_dataset)
    
    typiclust_accuracies.append(eval_queries(tc_queries))
    random_accuracies.append(eval_queries(rand_queries))

plt.figure(figsize=(8, 5))
plt.plot(budgets, typiclust_accuracies, marker='o', label='TypiClust', color='blue', linewidth=2)
plt.plot(budgets, random_accuracies, marker='s', label='Random Baseline', color='black', linestyle='--', linewidth=2)

plt.title('Low Budget Active Learning: TypiClust vs Random')
plt.xlabel('Cumulative Budget (Number of Labeled Examples)')
plt.ylabel('Test Accuracy (%)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, config['evaluation']['learning_curve_modified_filename'])) 
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

subset_size = config['visualization']['subset_size']
subset_idx = np.random.choice(len(embeddings), subset_size, replace=False)
emb_subset = embeddings[subset_idx]

tsne = TSNE(n_components=2, random_state=config['visualization']['random_state'])
emb_2d = tsne.fit_transform(emb_subset)

kmeans_viz = KMeans(n_clusters=config['visualization']['num_clusters_viz'], random_state=config['visualization']['random_state'], n_init=config['typiclust']['kmeans_n_init'])
cluster_labels_viz = kmeans_viz.fit_predict(emb_subset)

tc_queries_viz = get_typiclust_queries(emb_subset, B=10, max_clusters=config['typiclust']['max_clusters'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=cluster_labels_viz, cmap='tab20', s=10, alpha=0.6)

query_2d = emb_2d[tc_queries_viz]
plt.scatter(query_2d[:, 0], query_2d[:, 1], c='black', marker='x', s=150, linewidths=3, label='TypiClust Queries')

plt.title('t-SNE Visualization of Feature Space and Selected Queries')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, config['visualization']['tsne_modified_filename']))
plt.show()

--- Final Quantitative Evaluation (B=10 to 100) ---
Mean Accuracy (Original): 36.40%
Mean Accuracy (Modified): 36.56%
----------------------------------------
T-statistic: 0.2555
P-value:     0.8041
----------------------------------------
